# 08 — Classification Testing (Eval A + Eval B)

Evaluates a trained cls experiment twice on each fold's test set:

| Variant | Mask source | Output dir |
|---|---|---|
| **Eval A** | GT masks from `data/.../processed/masks/` | `eval_gt/` |
| **Eval B** | Predicted masks from a seg experiment | `eval_pred__<seg_exp>/` |

Both evals use the **same trained checkpoint** and the same fold CSVs.
The gap (Eval A macro F1 − Eval B macro F1) is the project's headline contribution.

**Prerequisites:**
- Trained cls checkpoints on Drive (NB07): `outputs/checkpoints/classification/.../fold_k/best_model.pt`
- Seg predictions on Drive (NB05): `outputs/predictions/segmentation/<dataset>/<SEG_EXPERIMENT>/fold_k/`
- Seg checkpoints on Drive (needed for manifest verification): `outputs/checkpoints/segmentation/<dataset>/<SEG_EXPERIMENT>/fold_k/best_model.pt`

**Outputs** (synced to Drive by Cell 14):
```
outputs/tables/classification/<dataset>/<cls_exp>/
    eval_gt/
        fold_k_per_image.csv, fold_k_metrics.csv, fold_k_manifest.json
        cv_results.csv, cv_summary.csv, cv_summary_enriched.csv
        cv_confusion.csv, cv_by_class.csv, cv_per_image.csv
    eval_pred__<seg_exp>/  (same structure)
outputs/figures/classification/<dataset>/<cls_exp>/
    eval_gt/confusion_matrix.png
    eval_pred__<seg_exp>/confusion_matrix.png
```

**Runtime:** GPU (T4 or better). Inference is fast — both evals typically finish in < 10 minutes.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists('/content/senior_project'):
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    url = 'https://github.com/salemaker47/senior_project.git'
    if token:
        url = url.replace('https://', f'https://{token}@', 1)
    os.system(f'git clone {url} /content/senior_project')
if '/content/senior_project' not in sys.path:
    sys.path.insert(0, '/content/senior_project')

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url='https://github.com/salemaker47/senior_project.git',
)
print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print(f'REPO_ROOT:  {REPO_ROOT}')

## Cell 3 — Config

Edit `RECIPE`, `DATASET`, `SPLIT_SCHEME`, and `SEG_EXPERIMENT` to match the experiments you want to evaluate.

**Knobs:**
- `RECIPE` / `DATASET` / `SPLIT_SCHEME` — must match what was used in NB07
- `SEG_EXPERIMENT` — seg experiment whose predicted masks to use for Eval B (from NB05)
- `FOLD_TO_RUN` / `RUN_ALL_FOLDS` — fold control (same as NB07)

In [ ]:
from configs.cls.reference_experiments import get_experiment, REFERENCE_RECIPES

# ----- The 3 axes of a run -----
RECIPE        = 'cls01_resnet50'                          # must match NB07
DATASET       = 'figshare'                                # only figshare for classification
SPLIT_SCHEME  = 'image_level'                             # must match NB07

# Seg experiment whose predicted masks to use for Eval B
SEG_EXPERIMENT = '07_unetpp_effb4_dicebce_image_level'    # best seg model

# ----- Fold control (same as NB07) -----
FOLD_TO_RUN   = 1       # used when RUN_ALL_FOLDS = False
RUN_ALL_FOLDS = False   # flip to True to run all 5 folds

# ----- Inference knobs -----
BATCH_SIZE   = 32
NUM_WORKERS  = 2
DEVICE       = 'cuda'   # 'cpu' if no GPU available

# ---------------------------------------------------------------------------
EXPERIMENT = get_experiment(
    RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1,
)

assert EXPERIMENT['task'] == 'classification'
assert EXPERIMENT['dataset'] in ('figshare',), \
    f'Classification only supported on figshare, got {EXPERIMENT["dataset"]!r}'
assert EXPERIMENT['split_scheme'] in ('image_level', 'patient_level')

folds_to_run = list(range(1, 6)) if RUN_ALL_FOLDS else [FOLD_TO_RUN]

print(f"EXPERIMENT:    {EXPERIMENT['name']}")
print(f"  model:        {EXPERIMENT['model_name']}")
print(f"  dataset:      {EXPERIMENT['dataset']}")
print(f"  split_scheme: {EXPERIMENT['split_scheme']}")
print(f"  patch_size:   {EXPERIMENT['patch_size']}")
print(f"SEG_EXPERIMENT: {SEG_EXPERIMENT}")
print(f"folds_to_run:  {folds_to_run}")

## Cell 4 — Copy data + checkpoints + seg predictions to local SSD

In [ ]:
import shutil
from src.notebook_setup import copy_to_local

# 1. Copy dataset (images + masks + splits CSVs)
LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT['dataset']])
print(f'LOCAL_ROOT: {LOCAL_ROOT}')

# 2. Copy cls checkpoints from Drive -> local SSD
drive_cls_ckpts = DRIVE_ROOT / 'outputs' / 'checkpoints' / 'classification' \
                  / EXPERIMENT['dataset'] / EXPERIMENT['name']
local_cls_ckpts = LOCAL_ROOT / 'outputs' / 'checkpoints' / 'classification' \
                  / EXPERIMENT['dataset'] / EXPERIMENT['name']

if drive_cls_ckpts.exists():
    local_cls_ckpts.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(drive_cls_ckpts), str(local_cls_ckpts), dirs_exist_ok=True)
    print(f'Copied cls checkpoints: {drive_cls_ckpts}')
else:
    print(f'WARNING: cls checkpoints not found at {drive_cls_ckpts}')
    print('         Run NB07 first to train the classification model.')

# 3. Copy seg predictions from Drive -> local SSD (needed for Eval B + verify)
drive_seg_preds = DRIVE_ROOT / 'outputs' / 'predictions' / 'segmentation' \
                  / EXPERIMENT['dataset'] / SEG_EXPERIMENT
local_seg_preds = LOCAL_ROOT / 'outputs' / 'predictions' / 'segmentation' \
                  / EXPERIMENT['dataset'] / SEG_EXPERIMENT

if drive_seg_preds.exists():
    local_seg_preds.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(drive_seg_preds), str(local_seg_preds), dirs_exist_ok=True)
    print(f'Copied seg predictions: {drive_seg_preds}')
else:
    print(f'WARNING: seg predictions not found at {drive_seg_preds}')
    print('         Run NB05 first to generate segmentation predictions.')

# 4. Copy seg checkpoints from Drive -> local SSD (needed by verify_seg_predictions_match)
drive_seg_ckpts = DRIVE_ROOT / 'outputs' / 'checkpoints' / 'segmentation' \
                  / EXPERIMENT['dataset'] / SEG_EXPERIMENT
local_seg_ckpts = LOCAL_ROOT / 'outputs' / 'checkpoints' / 'segmentation' \
                  / EXPERIMENT['dataset'] / SEG_EXPERIMENT

if drive_seg_ckpts.exists():
    local_seg_ckpts.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(drive_seg_ckpts), str(local_seg_ckpts), dirs_exist_ok=True)
    print(f'Copied seg checkpoints: {drive_seg_ckpts}')
else:
    print(f'WARNING: seg checkpoints not found at {drive_seg_ckpts}')
    print('         verify_seg_predictions_match() will be skipped (legacy mode).')

## Cell 5 — Sanity check: experiment config

Verify that the saved `experiment_config.json` on Drive matches the EXPERIMENT dict above.
A mismatch means NB07 was run with different settings.

In [ ]:
import json
from src.file_utils import experiment_paths, fold_split_csv_paths

mismatches = []
for fold in folds_to_run:
    paths = experiment_paths(
        LOCAL_ROOT,
        task=EXPERIMENT['task'], dataset=EXPERIMENT['dataset'],
        experiment_name=EXPERIMENT['name'], fold=fold,
    )
    cfg_path = paths['experiment_config_json']
    if not cfg_path.exists():
        print(f'  fold {fold}: MISSING experiment_config.json — cls checkpoint not found')
        mismatches.append(fold)
        continue

    with open(cfg_path) as f:
        saved = json.load(f)
    saved_exp = saved.get('EXPERIMENT', {})

    checks = [
        ('split_scheme', EXPERIMENT['split_scheme']),
        ('dataset',      EXPERIMENT['dataset']),
        ('recipe',       EXPERIMENT['recipe']),
        ('model_name',   EXPERIMENT['model_name']),
    ]
    fold_ok = True
    for key, expected in checks:
        actual = saved_exp.get(key)
        if actual != expected:
            print(f'  fold {fold}: mismatch {key}: saved={actual!r} vs config={expected!r}')
            mismatches.append(fold)
            fold_ok = False
    if fold_ok:
        print(f'  fold {fold}: OK  (model={saved_exp.get("model_name")}  '
              f'scheme={saved_exp.get("split_scheme")})')

if not mismatches:
    print('All config sanity checks passed.')
else:
    print(f'\nFolds with issues: {mismatches}')
    print('Re-run NB07 for the affected folds before continuing.')

---
## Cell 6 — evaluate_one_fold_cls helper

Defines the function used by both Eval A (Cell 7) and Eval B (Cell 10).
The only difference between the two variants is `mask_source`.

In [ ]:
import torch
import pandas as pd
from pathlib import Path

from src.data_utils       import load_metadata
from src.cls_test_utils   import load_cls_model_from_pt, evaluate_fold_cls
from src.file_utils       import (
    experiment_paths, fold_split_csv_paths,
    cls_eval_paths, seg_predictions_dir,
)
from src.train_utils      import gather_repro_metadata


def evaluate_one_fold_cls(
    fold: int,
    experiment: dict,
    mask_source: str = 'gt',
) -> dict:
    """
    Load the best cls checkpoint for `fold`, run inference on the test set,
    compute metrics, and save per-fold tables to the appropriate eval_dir.

    Parameters
    ----------
    fold         : fold number (1–5)
    experiment   : EXPERIMENT dict (from get_experiment)
    mask_source  : 'gt' for Eval A, 'predicted' for Eval B

    Returns
    -------
    dict from evaluate_fold_cls: {per_image_df, fold_metrics,
                                   confusion_matrix, manifest_path, manifest}
    """
    experiment = dict(experiment)
    experiment['fold'] = fold

    # ----- File paths -----
    paths = experiment_paths(
        LOCAL_ROOT, task=experiment['task'],
        dataset=experiment['dataset'],
        experiment_name=experiment['name'], fold=fold,
    )
    csv_paths = fold_split_csv_paths(
        LOCAL_ROOT, dataset=experiment['dataset'],
        scheme=experiment['split_scheme'], fold=fold,
    )
    eval_paths = cls_eval_paths(
        LOCAL_ROOT, dataset=experiment['dataset'],
        cls_experiment_name=experiment['name'],
        mask_source=mask_source,
        seg_experiment_name=SEG_EXPERIMENT if mask_source == 'predicted' else None,
    )

    # ----- Test data -----
    test_df = load_metadata(csv_paths['test'])
    print(f'  fold {fold}: test={len(test_df):>5} images  '
          f'({test_df["tumor_class"].value_counts().to_dict()})')

    # ----- Load checkpoint -----
    pt_path = paths['best_model']
    if not pt_path.exists():
        raise FileNotFoundError(
            f'Cls checkpoint not found: {pt_path}\n'
            f'Run NB07 fold {fold} first.'
        )
    model = load_cls_model_from_pt(
        pt_path=pt_path,
        model_name=experiment['model_name'],
        num_classes=experiment['num_classes'],
        device=DEVICE,
    )

    # ----- Predicted masks dir (Eval B only) -----
    predictions_dir = None
    if mask_source == 'predicted':
        predictions_dir = seg_predictions_dir(
            LOCAL_ROOT, dataset=experiment['dataset'],
            seg_experiment_name=SEG_EXPERIMENT, fold=fold,
        )

    # ----- Evaluate -----
    result = evaluate_fold_cls(
        model=model,
        test_df=test_df,
        project_root=LOCAL_ROOT,
        eval_dir=eval_paths['eval_dir'],
        fold=fold,
        experiment_name=experiment['name'],
        dataset=experiment['dataset'],
        split_scheme=experiment['split_scheme'],
        checkpoint_path=pt_path,
        test_csv_path=csv_paths['test'],
        model_name=experiment['model_name'],
        mask_source=mask_source,
        seg_experiment_name=SEG_EXPERIMENT if mask_source == 'predicted' else None,
        predictions_dir=predictions_dir,
        image_size=experiment['patch_size'],
        padding_frac=experiment['padding_frac'],
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=DEVICE,
    )

    # ----- Save per-fold CSVs -----
    tables_dir = eval_paths['tables_dir']
    result['per_image_df'].to_csv(
        tables_dir / f'fold_{fold}_per_image.csv', index=False
    )
    metrics_df = pd.DataFrame([result['fold_metrics']])
    metrics_df.insert(0, 'fold', fold)
    metrics_df.to_csv(tables_dir / f'fold_{fold}_metrics.csv', index=False)

    mf1 = result['fold_metrics']['macro_f1']
    acc  = result['fold_metrics']['accuracy']
    print(f'  fold {fold}: macro_f1={mf1:.4f}  accuracy={acc:.4f}  '
          f'[{mask_source}]')

    # free GPU memory before the next fold
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


print('evaluate_one_fold_cls defined.')

---
## Cell 7 — Eval A fold loop (GT masks)

In [ ]:
import gc

fold_summaries_a = []

print(f"{'='*70}")
print(f"  EVAL A — GT masks  ({EXPERIMENT['name']} on {EXPERIMENT['dataset']})")
print(f"{'='*70}")

for fold in folds_to_run:
    print(f"\n--- fold {fold} ---")
    try:
        r = evaluate_one_fold_cls(fold, EXPERIMENT, mask_source='gt')
        fold_summaries_a.append(r)
    except Exception as e:
        print(f'  fold {fold} FAILED: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
    finally:
        gc.collect()

print(f"\n{'='*70}")
print(f"  Eval A complete: {len(fold_summaries_a)}/{len(folds_to_run)} folds")
print(f"{'='*70}")

## Cell 8 — Eval A aggregation

In [ ]:
from src.cls_eval_utils  import aggregate_cv_results_cls
from src.file_utils      import cls_eval_paths
from src.train_utils     import gather_repro_metadata
from IPython.display     import display

if not fold_summaries_a:
    print('No Eval A fold summaries available — check Cell 7 for errors.')
else:
    repro_meta = gather_repro_metadata(repo_root=REPO_ROOT)

    cv_a = aggregate_cv_results_cls(
        fold_summaries=fold_summaries_a,
        experiment_name=EXPERIMENT['name'],
        dataset=EXPERIMENT['dataset'],
        split_scheme=EXPERIMENT['split_scheme'],
        mask_source='gt',
        seg_experiment_name=None,
        repro_metadata=repro_meta,
    )

    eval_paths_a = cls_eval_paths(
        LOCAL_ROOT, dataset=EXPERIMENT['dataset'],
        cls_experiment_name=EXPERIMENT['name'],
        mask_source='gt',
    )
    tables_dir_a = eval_paths_a['tables_dir']

    cv_a['cv_results'].to_csv(tables_dir_a / 'cv_results.csv',           index=False)
    cv_a['cv_summary'].to_csv(tables_dir_a / 'cv_summary.csv',           index=False)
    cv_a['cv_summary_enriched'].to_csv(tables_dir_a / 'cv_summary_enriched.csv', index=False)
    cv_a['cv_confusion'].to_csv(tables_dir_a / 'cv_confusion.csv')
    cv_a['cv_by_class'].to_csv(tables_dir_a / 'cv_by_class.csv',         index=False)
    cv_a['cv_per_image'].to_csv(tables_dir_a / 'cv_per_image.csv',       index=False)

    print('Eval A — cross-fold results:')
    display(cv_a['cv_results'][[
        'fold', 'n_images', 'macro_f1', 'accuracy',
        'f1_meningioma', 'f1_glioma', 'f1_pituitary'
    ]].style.format({
        'macro_f1': '{:.4f}', 'accuracy': '{:.4f}',
        'f1_meningioma': '{:.4f}', 'f1_glioma': '{:.4f}', 'f1_pituitary': '{:.4f}'
    }))
    print()
    print(f"Eval A macro F1: {cv_a['cv_summary']['macro_f1_mean'].iloc[0]:.4f} "
          f"± {cv_a['cv_summary']['macro_f1_std'].iloc[0]:.4f}")
    print(f'Tables written to: {tables_dir_a}')

---
## Cell 9 — Eval B: verify seg predictions

Before running Eval B, verify that the predicted masks on local SSD were generated
by the exact checkpoint and test CSV recorded in each fold's manifest.

A hash mismatch means the seg experiment was retrained after the predictions were
written — re-run NB05 to regenerate predictions before continuing.

In [ ]:
from src.file_utils import verify_seg_predictions_match, fold_split_csv_paths

verify_ok_folds = []
verify_skip_folds = []

for fold in folds_to_run:
    seg_ckpt_path = (
        LOCAL_ROOT / 'outputs' / 'checkpoints' / 'segmentation'
        / EXPERIMENT['dataset'] / SEG_EXPERIMENT / f'fold_{fold}' / 'best_model.pt'
    )
    test_csv_path = fold_split_csv_paths(
        LOCAL_ROOT, dataset=EXPERIMENT['dataset'],
        scheme=EXPERIMENT['split_scheme'], fold=fold,
    )['test']

    if not seg_ckpt_path.exists():
        print(f'  fold {fold}: seg checkpoint not found — skipping hash check  '
              f'(proceeding with unverified predictions)')
        verify_skip_folds.append(fold)
        continue

    try:
        manifest = verify_seg_predictions_match(
            LOCAL_ROOT, EXPERIMENT['dataset'],
            SEG_EXPERIMENT, fold,
            seg_ckpt_path, test_csv_path,
        )
        legacy = manifest.get('legacy', False)
        status = 'legacy (hash skip)' if legacy else 'OK'
        print(f'  fold {fold}: {status}  '
              f'(n_predictions={manifest.get("n_predictions")})')
        verify_ok_folds.append(fold)
    except ValueError as e:
        print(f'  fold {fold}: HASH MISMATCH — {e}')
        print(f'           Re-run NB05 for {SEG_EXPERIMENT} fold {fold} to fix.')

print()
print(f'Verified: {verify_ok_folds}')
if verify_skip_folds:
    print(f'Skipped (no seg checkpoint): {verify_skip_folds}')

## Cell 10 — Eval B fold loop (predicted masks)

In [ ]:
fold_summaries_b = []

print(f"{'='*70}")
print(f"  EVAL B — predicted masks  (seg: {SEG_EXPERIMENT})")
print(f"  cls experiment: {EXPERIMENT['name']} on {EXPERIMENT['dataset']}")
print(f"{'='*70}")

for fold in folds_to_run:
    print(f"\n--- fold {fold} ---")
    try:
        r = evaluate_one_fold_cls(fold, EXPERIMENT, mask_source='predicted')
        fold_summaries_b.append(r)
    except Exception as e:
        print(f'  fold {fold} FAILED: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
    finally:
        gc.collect()

print(f"\n{'='*70}")
print(f"  Eval B complete: {len(fold_summaries_b)}/{len(folds_to_run)} folds")
print(f"{'='*70}")

## Cell 11 — Eval B aggregation

In [ ]:
if not fold_summaries_b:
    print('No Eval B fold summaries available — check Cell 10 for errors.')
else:
    cv_b = aggregate_cv_results_cls(
        fold_summaries=fold_summaries_b,
        experiment_name=EXPERIMENT['name'],
        dataset=EXPERIMENT['dataset'],
        split_scheme=EXPERIMENT['split_scheme'],
        mask_source='predicted',
        seg_experiment_name=SEG_EXPERIMENT,
        repro_metadata=gather_repro_metadata(repo_root=REPO_ROOT),
    )

    eval_paths_b = cls_eval_paths(
        LOCAL_ROOT, dataset=EXPERIMENT['dataset'],
        cls_experiment_name=EXPERIMENT['name'],
        mask_source='predicted',
        seg_experiment_name=SEG_EXPERIMENT,
    )
    tables_dir_b = eval_paths_b['tables_dir']

    cv_b['cv_results'].to_csv(tables_dir_b / 'cv_results.csv',           index=False)
    cv_b['cv_summary'].to_csv(tables_dir_b / 'cv_summary.csv',           index=False)
    cv_b['cv_summary_enriched'].to_csv(tables_dir_b / 'cv_summary_enriched.csv', index=False)
    cv_b['cv_confusion'].to_csv(tables_dir_b / 'cv_confusion.csv')
    cv_b['cv_by_class'].to_csv(tables_dir_b / 'cv_by_class.csv',         index=False)
    cv_b['cv_per_image'].to_csv(tables_dir_b / 'cv_per_image.csv',       index=False)

    print('Eval B — cross-fold results:')
    display(cv_b['cv_results'][[
        'fold', 'n_images', 'macro_f1', 'accuracy',
        'f1_meningioma', 'f1_glioma', 'f1_pituitary'
    ]].style.format({
        'macro_f1': '{:.4f}', 'accuracy': '{:.4f}',
        'f1_meningioma': '{:.4f}', 'f1_glioma': '{:.4f}', 'f1_pituitary': '{:.4f}'
    }))
    print()
    print(f"Eval B macro F1: {cv_b['cv_summary']['macro_f1_mean'].iloc[0]:.4f} "
          f"± {cv_b['cv_summary']['macro_f1_std'].iloc[0]:.4f}")
    print(f'Tables written to: {tables_dir_b}')

## Cell 12 — Confusion matrix plots

In [ ]:
from src.cls_vis_utils import plot_confusion_matrix
import matplotlib.pyplot as plt

has_a = fold_summaries_a
has_b = fold_summaries_b

n_panels = int(bool(has_a)) + int(bool(has_b))
if n_panels == 0:
    print('No fold summaries available for confusion matrix plots.')
else:
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]

    ax_idx = 0
    if has_a:
        plot_confusion_matrix(
            cv_a['cv_confusion'],
            title=f"Eval A (GT masks)\n{EXPERIMENT['name']}",
            ax=axes[ax_idx],
        )
        fig_dir_a = cls_eval_paths(
            LOCAL_ROOT, EXPERIMENT['dataset'], EXPERIMENT['name'], 'gt'
        )['figures_dir']
        fig_dir_a.mkdir(parents=True, exist_ok=True)
        ax_idx += 1

    if has_b:
        plot_confusion_matrix(
            cv_b['cv_confusion'],
            title=f"Eval B (pred masks)\n{SEG_EXPERIMENT}",
            ax=axes[ax_idx],
        )
        fig_dir_b = cls_eval_paths(
            LOCAL_ROOT, EXPERIMENT['dataset'], EXPERIMENT['name'],
            'predicted', SEG_EXPERIMENT,
        )['figures_dir']
        fig_dir_b.mkdir(parents=True, exist_ok=True)

    fig.suptitle(
        f"{EXPERIMENT['name']}  ·  {EXPERIMENT['dataset']}  "
        f"— confusion matrices (sum across folds)",
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

    if has_a:
        out_a = fig_dir_a / 'confusion_matrix.png'
        fig.savefig(out_a, dpi=120, bbox_inches='tight')
        print(f'saved {out_a}')
    if has_b:
        out_b = fig_dir_b / 'confusion_matrix.png'
        fig.savefig(out_b, dpi=120, bbox_inches='tight')
        print(f'saved {out_b}')

---
## Cell 13 — Gap computation (Eval A − Eval B)

The headline contribution of the project: how much does using predicted masks
instead of GT masks degrade classification performance?

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if not fold_summaries_a or not fold_summaries_b:
    print('Both Eval A and Eval B must be complete before computing the gap.')
    print(f'  Eval A folds: {len(fold_summaries_a)}')
    print(f'  Eval B folds: {len(fold_summaries_b)}')
else:
    # Build aligned per-fold table (inner join on fold)
    res_a = cv_a['cv_results'][['fold', 'macro_f1', 'accuracy']].rename(
        columns={'macro_f1': 'eval_a_macro_f1', 'accuracy': 'eval_a_accuracy'}
    )
    res_b = cv_b['cv_results'][['fold', 'macro_f1', 'accuracy']].rename(
        columns={'macro_f1': 'eval_b_macro_f1', 'accuracy': 'eval_b_accuracy'}
    )
    gap_df = res_a.merge(res_b, on='fold', how='inner')
    gap_df['gap_macro_f1'] = gap_df['eval_a_macro_f1'] - gap_df['eval_b_macro_f1']
    gap_df['gap_accuracy']  = gap_df['eval_a_accuracy']  - gap_df['eval_b_accuracy']

    # Summary statistics
    n_folds  = len(gap_df)
    ddof     = 1 if n_folds > 1 else 0

    a_f1_mean  = gap_df['eval_a_macro_f1'].mean()
    a_f1_std   = gap_df['eval_a_macro_f1'].std(ddof=ddof)
    b_f1_mean  = gap_df['eval_b_macro_f1'].mean()
    b_f1_std   = gap_df['eval_b_macro_f1'].std(ddof=ddof)
    gap_f1_mean = gap_df['gap_macro_f1'].mean()
    gap_f1_std  = gap_df['gap_macro_f1'].std(ddof=ddof)

    a_acc_mean  = gap_df['eval_a_accuracy'].mean()
    a_acc_std   = gap_df['eval_a_accuracy'].std(ddof=ddof)
    b_acc_mean  = gap_df['eval_b_accuracy'].mean()
    b_acc_std   = gap_df['eval_b_accuracy'].std(ddof=ddof)
    gap_acc_mean = gap_df['gap_accuracy'].mean()
    gap_acc_std  = gap_df['gap_accuracy'].std(ddof=ddof)

    print(f"{'='*60}")
    print(f"  {EXPERIMENT['name']}  —  {n_folds}-fold gap summary")
    print(f"  Seg: {SEG_EXPERIMENT}")
    print(f"{'='*60}")
    print(f"  Metric         Eval A              Eval B              Gap (A−B)")
    print(f"  {'─'*54}")
    print(f"  macro F1   {a_f1_mean:.4f} ± {a_f1_std:.4f}   "
          f"{b_f1_mean:.4f} ± {b_f1_std:.4f}   "
          f"{gap_f1_mean:+.4f} ± {gap_f1_std:.4f}")
    print(f"  accuracy   {a_acc_mean:.4f} ± {a_acc_std:.4f}   "
          f"{b_acc_mean:.4f} ± {b_acc_std:.4f}   "
          f"{gap_acc_mean:+.4f} ± {gap_acc_std:.4f}")
    print(f"{'='*60}")

    print('\nPer-fold breakdown:')
    display(gap_df.style.format({
        'eval_a_macro_f1': '{:.4f}', 'eval_b_macro_f1': '{:.4f}', 'gap_macro_f1': '{:+.4f}',
        'eval_a_accuracy': '{:.4f}', 'eval_b_accuracy': '{:.4f}', 'gap_accuracy':  '{:+.4f}',
    }))

    # Save gap table
    from src.file_utils import experiment_root_paths
    exp_root = experiment_root_paths(
        LOCAL_ROOT, task=EXPERIMENT['task'],
        dataset=EXPERIMENT['dataset'],
        experiment_name=EXPERIMENT['name'],
    )
    gap_csv = exp_root['tables'] / f"{EXPERIMENT['name']}_gap_{SEG_EXPERIMENT}.csv"
    gap_df.to_csv(gap_csv, index=False)
    print(f'\nGap table saved to: {gap_csv}')

## Cell 14 — Sync to Drive

In [ ]:
from src.notebook_setup import sync_outputs_to_drive

sync_outputs_to_drive(
    drive_root=DRIVE_ROOT, local_root=LOCAL_ROOT,
    task=EXPERIMENT['task'], dataset=EXPERIMENT['dataset'],
    experiment_name=EXPERIMENT['name'],
    categories=['tables', 'figures'],
)
print('sync complete')

In [ ]:
SYNC_OK = True   # set manually based on the sync cell above
if SYNC_OK:
    from google.colab import runtime
    print('Disconnecting runtime in 3 seconds...')
    import time; time.sleep(3)
    runtime.unassign()
else:
    print('SYNC_OK is False — staying connected.')